# Controlled Poisson diagnostics for facial reconstruction

This notebook runs a **non-authoritative** A/B/C/D experiment on one permanent COLMAP stereo-fusion point cloud. It is designed to determine whether rough facial geometry originates primarily from the COLMAP normals, fused XYZ outliers/thickness, or Poisson reconstruction.

The notebook never changes `face_dense_fused.ply`, `face_mesh_raw.ply`, `face_geometry.ply`, landmarks, measurements, geometry IDs, or `case.json`. All outputs are written to a new diagnostic run directory.

## 1. Select an appropriate Colab runtime

Use a **High-RAM** runtime when available. A T4 runtime is acceptable, but Open3D's Poisson implementation runs on the CPU; the GPU does not accelerate this experiment. Every additional configured depth creates four more meshes and takes substantially longer than the primary four-variant run.

In [ ]:
import os
import platform
import shutil

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
disk = shutil.disk_usage("/content")
print(f"Python: {platform.python_version()}")
print(f"RAM: {ram_gb:.1f} GB")
print(f"Free /content storage: {disk.free / 1e9:.1f} GB")
if ram_gb < 20:
    print("WARNING: A High-RAM runtime is recommended for a million-point depth-9 run.")

## 2. Mount Google Drive

The completed reconstruction case must already be stored in Drive and must contain `face_dense_fused.ply`, `scale.json`, and `case.json`. `landmarks.json` is optional but required for anatomical ROI diagnostics.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Install the diagnostic code

This notebook does not install or run COLMAP because it starts from the already preserved stereo-fusion artifact. The repository is installed without its unrelated capture dependencies, followed by the explicit numerical and geometry dependencies needed here.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Daml4Yilmaz/rhino-poc.git"
REPOSITORY = Path("/content/rhino-poc")

def run(command):
    command = [str(value) for value in command]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, check=True)

if (REPOSITORY / ".git").is_dir():
    run(["git", "-C", REPOSITORY, "pull", "--ff-only"])
else:
    run(["git", "clone", "--quiet", REPOSITORY_URL, REPOSITORY])

run([sys.executable, "-m", "pip", "install", "--quiet",
     "numpy>=1.26,<3", "scipy>=1.11,<2", "open3d>=0.18,<1",
     "trimesh>=4,<5", "pandas>=2,<3"])
run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", REPOSITORY])

sys.path.insert(0, str(REPOSITORY))
print("Diagnostic environment is ready.")

## 4. Select and validate one completed case

Edit only `CASE_DIR` and `EXPERIMENT_NAME`. Use a new experiment name for every run; the diagnostic runner refuses to mix or overwrite previous results.

In [ ]:
import json

CASE_DIR = Path("/content/drive/MyDrive/rhino-poc-results/vaka_003_stray/final")
EXPERIMENT_NAME = "poisson_abcd_003"
OUTPUT_DIR = CASE_DIR / "diagnostics" / EXPERIMENT_NAME

required = [CASE_DIR / "face_dense_fused.ply", CASE_DIR / "scale.json", CASE_DIR / "case.json"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required case artifacts:\n" + "\n".join(missing))
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(f"{OUTPUT_DIR} is not empty. Choose a new EXPERIMENT_NAME.")

manifest = json.loads((CASE_DIR / "case.json").read_text())
scale_document = json.loads((CASE_DIR / "scale.json").read_text())
production_depth = int(manifest.get("stages", {}).get("mvs", {}).get("parameters", {}).get("poisson_depth", 9))
print(f"Case: {CASE_DIR.name}")
print(f"Fused cloud: {(CASE_DIR / 'face_dense_fused.ply').stat().st_size / 1e6:.1f} MB")
print(f"Metric scale: {scale_document['scale_mm_per_unit']:.6f} mm/unit")
print(f"Recorded production Poisson depth: {production_depth}")
print(f"Diagnostic output: {OUTPUT_DIR}")
if not (CASE_DIR / "landmarks.json").is_file():
    print("WARNING: landmarks.json is absent; global diagnostics will run, but anatomical ROIs will be reported unavailable.")

## 5. Configure the controlled experiment

The notebook first runs A/B/C/D at the recorded production depth. It then automatically runs the same four controlled variants at each configured sweep depth. The production depth is not repeated, and all other settings remain identical across variants and depths.

- A: original XYZ + original COLMAP normals
- B: original XYZ + explicitly recomputed/oriented normals
- C: conservatively filtered XYZ + retained COLMAP normals
- D: conservatively filtered XYZ + explicitly recomputed/oriented normals

No smoothing, template fitting, position averaging, texturing, or automatic promotion is performed.

In [ ]:
from poc.diagnostics.poisson import PoissonDiagnosticConfig

depth_sweep = (10, 11)
remaining_depths = tuple(depth for depth in depth_sweep if depth != production_depth)
depths = (production_depth, *remaining_depths)

config = PoissonDiagnosticConfig(
    poisson_depths=depths,
    production_poisson_depth=production_depth,
    poisson_trim_percent=4.0,
    normal_radius_mm=2.5,
    normal_max_nn=64,
    normal_fast_computation=False,
    orientation_neighbors=30,
    outlier_filter_neighbors=30,
    outlier_filter_std_ratio=3.0,
    maximum_removed_percent=5.0,
    normal_sample_count=50_000,
    normal_neighbor_count=12,
    roi_normal_sample_count=10_000,
    thickness_sample_count=2_500,
    thickness_radius_mm=2.0,
    thickness_max_nn=64,
    thickness_min_neighbors=12,
    random_seed=20260815,
)
config.validate()
print(config)

## 6. Run A/B/C/D

Progress is printed before normal estimation, filtering, diagnostics, and every Poisson variant. The production-depth A/B/C/D comparison completes first; the configured sweep depths follow automatically. The JSON report is updated after each mesh, so completed work remains inspectable if the Colab session disconnects. A failed run is marked `failed` with its exception.

In [ ]:
from poc.diagnostics.poisson import run_poisson_diagnostic

report_path = run_poisson_diagnostic(CASE_DIR, OUTPUT_DIR, config=config)
print(f"Report written to: {report_path}")

## 7. Review the mesh comparison

This table uses the persisted untextured meshes. Edge lengths are converted to millimetres with the case's recorded scale. Component counts are triangle-connected components.

In [ ]:
import pandas as pd

report = json.loads(report_path.read_text())
mesh_rows = []
for depth, variants in report["mesh_results"].items():
    for variant, metrics in variants.items():
        mesh_rows.append({
            "depth": int(depth), "variant": variant,
            "vertices": metrics["vertex_count"],
            "triangles": metrics["triangle_count"],
            "components": metrics["triangle_connected_component_count"],
            "largest_component_ratio": metrics["largest_component_ratio"],
            "boundary_edges": metrics["boundary_edge_count"],
            "watertight": metrics["watertight"],
            "median_edge_mm": metrics["median_edge_length_mm"],
            "p95_edge_mm": metrics["p95_edge_length_mm"],
            "elapsed_seconds": metrics["elapsed_seconds"],
        })
pd.DataFrame(mesh_rows).sort_values(["depth", "variant"])

## 8. Review normal coherence

The oriented angle includes sign flips; the plane angle treats `n` and `-n` as the same tangent plane. Reviewing both prevents orientation errors from being confused with local surface-direction noise.

In [ ]:
normal_rows = []
for variant, metrics in report["normal_statistics"].items():
    if metrics.get("status") != "available":
        continue
    oriented = metrics["oriented_neighbor_angle_degrees"]
    plane = metrics["unoriented_plane_angle_degrees"]
    normal_rows.append({
        "variant": variant, "samples": metrics["sample_count"],
        "oriented_median_deg": oriented["p50"],
        "oriented_p95_deg": oriented["p95"],
        "plane_median_deg": plane["p50"],
        "plane_p95_deg": plane["p95"],
        "dot_lt_0": metrics["fraction_dot_lt_0"],
        "dot_lt_minus_0_5": metrics["fraction_dot_lt_minus_0_5"],
        "dot_lt_minus_0_9": metrics["fraction_dot_lt_minus_0_9"],
    })
pd.DataFrame(normal_rows).sort_values("variant")

## 9. Review smooth-region thickness and normal metrics

ROI definitions are approximate reconstruction diagnostics tied to existing landmarks; they are not clinically validated anatomy. The chin is deliberately reported unavailable because the current landmark schema has no menton or pogonion.

In [ ]:
thickness_rows = []
for roi, metrics in report["roi_surface_thickness"].items():
    row = {"roi": roi, "status": metrics.get("status")}
    if metrics.get("status") == "available":
        residual = metrics["absolute_local_plane_residual_mm"]
        row.update({
            "roi_points": metrics["candidate_point_count"],
            "median_abs_mm": metrics["median_absolute_local_plane_residual_mm"],
            "mad_mm": metrics["signed_residual_mad_mm"],
            "p90_mm": residual["p90"], "p95_mm": residual["p95"],
            "p99_mm": residual["p99"],
        })
    thickness_rows.append(row)
display(pd.DataFrame(thickness_rows))

roi_normal_rows = []
for roi, roi_metrics in report["roi_normal_statistics"].items():
    for variant in ("A", "B", "C", "D"):
        metrics = roi_metrics.get(variant, {})
        if metrics.get("status") != "available":
            continue
        angles = metrics["oriented_neighbor_angle_degrees"]
        roi_normal_rows.append({
            "roi": roi, "variant": variant,
            "points": metrics["candidate_point_count"],
            "median_deg": angles["p50"], "p90_deg": angles["p90"],
            "p95_deg": angles["p95"], "p99_deg": angles["p99"],
            "dot_lt_0": metrics["fraction_dot_lt_0"],
        })
display(pd.DataFrame(roi_normal_rows))

## 10. Inspect connected components at every processing stage

This directly tests whether density trimming fragments the surface, whether largest-component cleanup runs, whether persistence changes connectivity, and whether Open3D and the QA-style Trimesh definition disagree.

In [ ]:
component_rows = []
for depth, variants in report["mesh_results"].items():
    for variant, metrics in variants.items():
        for stage, stage_metrics in metrics["processing_stage_components"].items():
            component_rows.append({
                "depth": int(depth), "variant": variant, "stage": stage,
                "components": stage_metrics["triangle_connected_component_count"],
                "largest_ratio": stage_metrics["largest_component_ratio"],
                "triangles": stage_metrics["triangle_count"],
            })
display(pd.DataFrame(component_rows))
print("Detected anomalies:")
for anomaly in report["connected_component_investigation"].get("anomalies", []):
    print("-", anomaly)
if not report["connected_component_investigation"].get("anomalies"):
    print("- None in this run")

## 11. Inspect untextured geometry interactively

The viewer is optional. Select one depth and variant at a time. For a formal visual comparison, export fixed-camera screenshots from the four persisted PLY files using identical camera and lighting settings; do not compare textured and untextured models.

In [ ]:
import open3d as o3d

VIEW_DEPTH = production_depth
VIEW_VARIANT = "A"  # A, B, C, or D
mesh_record = report["artifacts"]["meshes"][str(VIEW_DEPTH)][VIEW_VARIANT]
mesh_path = OUTPUT_DIR / mesh_record["path"]
mesh = o3d.io.read_triangle_mesh(str(mesh_path))
mesh.compute_vertex_normals()
mesh.paint_uniform_color([0.72, 0.72, 0.72])
o3d.visualization.draw_plotly([mesh], width=900, height=700)

## 12. Preserve and download the experiment

The output directory is already permanent in Drive. The archive step is optional and can be large, especially after a depth sweep.

In [ ]:
print(json.dumps(report["interpretation_guide"], indent=2))
print(f"Machine-readable report: {report_path}")
print(f"All diagnostic artifacts: {OUTPUT_DIR}")

CREATE_ZIP = False
if CREATE_ZIP:
    archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
    print(f"Archive created: {archive}")